---
title: "Re-assembly Starting Communities"
author: "Afra Salazar and Sara Mitri"
date: "November 2025"
format: 
  html:
    self-contained: true
    fig-dpi: 400
    theme: cosmo
    fontsize: 1.1rem
    linestretch: 1.4
    linkcolor: "#82a6b3"
    mainfont: "Source Sans Pro"
    
    # Code
    highlight-style: arrow (light)
    code-fold: true
    code-summary: "Show the code"
toc: true
jupyter: Julia
---

In [1]:
#| code-summary: "Loading necessary libraries and packages"

using DataFrames, JLD2, CSV, Dates, LinearAlgebra, Colors, Statistics, JSON3, StatsBase, Distributions

In [2]:
#| code-summary: "Custom functions"

function getDATAFRAME(data, dname, coln)
    DICT = Dict()
    DICT["iso_index"] = string.(data[!, :iso_index])

    for name in dname
        zerovector = zeros(Int, l)
        xx = zerovector[findall(x -> x == name, data[!, coln])] .= 1
        DICT[name] = xx.parent
    end

    dataframe = DataFrame(DICT)
        
    dataframe_collapsed = combine(groupby(dataframe, :iso_index), dname .=> sum)
    dataframe_collapsed[!,:iso_index] = convert.(String, dataframe_collapsed[!, :iso_index])
    
    sumnames = names(dataframe_collapsed) #Combining groups added novel column names with the suffix "sum". I am here extracting such names. 
    
    for sumname in sumnames[2:length(sumnames)] # All colapsed sum, whenever larger than zero, is set equal to 1.
        rep = findall(x -> x > 0, dataframe_collapsed[!, sumname])
        dataframe_collapsed[rep, sumname] .= 1
    end
    
    dataframe_collapsed_perm = permutedims(dataframe_collapsed, "iso_index")
    
    return dataframe_collapsed_perm

end


function removeDiagonal(a)
    i = CartesianIndices(a)
    n = size(a, 1)
    k = filter(x -> x.I[1] != x.I[2], i)
    b = reshape(a'[k], n - 1, n)'
    return(b)
end


function getCoverage(MCommunity, td)
    M = []

    for μ in MCommunity
        this_commmunity_coverage = []
        
        for s in μ
            this_species_coverage = findall(x -> x > 0, td[!, s]) 
            push!(this_commmunity_coverage, this_species_coverage...)
        end
    
        push!(M, this_commmunity_coverage)
    end
    
    return length.(unique.(M))    
end

#Measure community complementarity using the consumption and production profiles
function communityComplementarity(mC, mP)
    intersectionCom = []
    S = size(mC, 1)

    for δi_consumption in eachrow(mC)
        intersectionSi = []
    
        for δj in 1:S
            δj_production = mP[δj, :]
            δj_consumption = mC[δj, :]
        
            a = δi_consumption .+ δj_production
            b = length(findall(==(2), a)) / sum(δi_consumption)

            push!(intersectionSi, b)
        end
        push!(intersectionCom, intersectionSi)
    end
    
    a = hcat(intersectionCom...)' #reshape(vcat(intersectionCom...), S, S)
    b = removeDiagonal(a)
    
    return b
    
end


#Get community complementarity of each community in the METACommunity
function getComplementarity(METACommunity, ϕS, ϕP)
    mC = []
    mP = []

    for μ in METACommunity   #Access to the community array, each μ is an array of strings
        δC_community = []
        δP_community = []
        
        for s in μ           #Access to each string (microbe name) of the community μ
            δC_species = ϕS[!, s] 
            δP_species = ϕP[!, s] 
            push!(δC_community, δC_species)
            push!(δP_community, δP_species)
        end
    
        push!(mC, δC_community)
        push!(mP, δP_community)
    end
    
    δC_Metacommunity = []

    for m in mC 
        push!(δC_Metacommunity, hcat(m...)')
    end 
    
    δP_Metacommunity = []
    
    for m in mP 
        push!(δP_Metacommunity, hcat(m...)')
    end 
    
    complementarities = []
    
    for c in 1:length(δC_Metacommunity)
        this_comm_complementarity = communityComplementarity(δC_Metacommunity[c], δP_Metacommunity[c])
        push!(complementarities, this_comm_complementarity)
    end
    
    return complementarities
    
end


function getCompetition(METACommunity, substrates)
    M = []

    for μ in METACommunity
        δC_community = []
        
        for s in μ
            δC_species = substrates[!, s] 
            push!(δC_community, δC_species)
        end
    
        push!(M, δC_community)
    end

    δC_Metacommunity = []

    for m in M 
        push!(δC_Metacommunity, hcat(m...)')
    end 
    
    competitionIndex = []
    
    for mConsumption in δC_Metacommunity
        intersectionCom = []

        for δi_consumption in eachrow(mConsumption)
            intersectionSi = []
    
            for δj_consumption in eachrow(mConsumption)
                a = δi_consumption .+ δj_consumption
                b = length(findall(==(2), a)) / sum(δi_consumption)

                push!(intersectionSi, b)
            end
            push!(intersectionCom, intersectionSi)
        end
    
        a = hcat(intersectionCom...)'
        b = removeDiagonal(a)
        
        push!(competitionIndex, b)
        
    end
    
    return competitionIndex
end

#Splits the rows in the Product column of the original dataframe into several rows according to the separator "+"
function expandDataFrameProducts(DF) 
    
    result_df = DataFrame()
    for row in eachrow(DF)
        split_elements = split(row["Product"], " + ") #Split rule
        temp_df = DataFrame("Product" => split_elements)
    
        for col in ["Gene", "Genome", "Gene_Count", "Path", "Activity", "Trophic_level", "Substrate", "Enzymatic_Activity", "iso_index"]
            temp_df[!, col] = repeat([row[col]], length(split_elements))
        end
    
        result_df = vcat(result_df, temp_df)
    end
    
    return result_df
    
end

function expandDataFrameSubstrates(DF) #Splits the rows in the Substrate column of the original dataframe into several rows according to the separator "+"

    
    result_df = DataFrame()
    for row in eachrow(DF)
        split_elements = split(row["Substrate"], " + ") #Split rule
        temp_df = DataFrame("Substrate" => split_elements)
    
        for col in ["Gene", "Genome", "Gene_Count", "Path", "Activity", "Trophic_level", "Product", "Enzymatic_Activity", "iso_index"]
            temp_df[!, col] = repeat([row[col]], length(split_elements))
        end
    
        result_df = vcat(result_df, temp_df)
    end
    
    return result_df 
end

function getMutationSize(community::Vector)
    n = length(community)
    p = 0.1 #Probability of mutation per community member 
    mutsize = clamp(rand(Binomial(n, p)), 1, n)
    return mutsize
end

#The functions that simulates the re-assasembly method, efficiently exploring the community space via 'Species mutations'
function mutateCommunities(METACommunity, METAPool) #MetaCommunity is the Array of Communities to be mutated, METAPool is the Array of all combinations made throught-out the whole experiment
    Communities = sort.(METACommunity)
    ηMETACommunity = []
    
    for community in Communities
        community = deepcopy(unique(community))
        oricom = deepcopy(community) #Original community
        μtype = sample([0, 1, 2]) #Mutation types: 0 Deletion, 1 Addition, 2 Substitution
        μsize = getMutationSize(community)
        oriϕcommunity = [sum(ϕ_levels_collapsed_perm[!, community][i, :]) for i in 1:τ]
        
        #Mutate the community while it is the same as the original community or the community already exists in the pool

        while (community == oricom) || sort(community) in sort.(METAPool)
            #Deletion
            if μtype == 0 && length(community) > 3 && length(community) > μsize
                a = sample(community, μsize, replace = true)
                μindex = findall(x -> x in a, community)
                tempcommunity = [community[i] for i in 1:length(community) if i ∉ μindex] #Creates a copy of the community removing the element in μindex
                any(x -> x == 0, tempcommunity) > 0 ? 0 : deleteat!(community, μindex)
            
            #Addition
            elseif μtype == 1 && (length(community) + μsize) < (length(speciesnames) - 1)
                μnames = setdiff(speciesnames, community) #x[x .∉ Ref(y)] 
                a = sample(μnames, μsize, replace = true)
                append!(community, a)
            
            #Substitution
            elseif μtype == 2 && (length(community) + μsize) < (length(speciesnames) - 1) && length(community) > μsize
                a = sample(community, μsize, replace = true)
                μindex = findall(x -> x in a, community) #Deleted community members
                
                μnames = setdiff(speciesnames, community) #x[x .∉ Ref(y)] #
                b = sample(μnames, μsize, replace = true) #Added community members
                
                tempcommunity = [community[i] for i in 1:length(community) if i ∉ μindex]
                append!(tempcommunity, b)
                
                any(x -> x == 0, tempcommunity) > 0 ? 0 : (deleteat!(community, μindex);append!(community, b))
            end
            
            #Change μtype (some types of mutations are impossible some communities and if not changed, the loop keeps running indefinitely)
            μtype = sample([0, 1, 2]) #Mutation types: 0 Deletion, 1 Addition, 2 Substitution
        end
        
        push!(ηMETACommunity, unique(community))

    end  
    
    return ηMETACommunity
end

function reAssembly(aCommunities, mem) #Communities to re-assemble, METACommunityPool Members to ensure uniqueness
    #Roulette wheel selection
    sh = aCommunities.CommunityScore .- abs(minimum(aCommunities.CommunityScore))
    wei = sh ./ sum(sh)
    samplingIndexes = sample(1:length(wei), Weights(wei), 63)
    motherID = aCommunities.CommunityID[samplingIndexes]

    communitiesforSampling = aCommunities.Members[samplingIndexes] #Selected communities to re-assemble/mutate
    
    #Create novel, mutated, communities
    ηMETACommunity = mutateCommunities(communitiesforSampling, mem.Members)
    ηMETACommunity = sort.(ηMETACommunity)

    return ηMETACommunity, motherID
end

function exportMetacommunity(METACommunity, METACommunityPool, ψ, pathcoverage, motherID, round)
    a = nrow(METACommunityPool) + 1
    b = nrow(METACommunityPool) + length(METACommunity)
    
    ExportDF = DataFrame(CommunityID = ["com$(i)" for i in a:b], Members = [parse.(Int, i) for i in METACommunity], MotherID = motherID, FunIntegration = ψ, Coverage = pathcoverage, CommunityScore = fill(missing, length(METACommunity)), Round = round)
    
    timenow = Dates.format(now(), "yyyy.mm.dd")
    CSV.write("$(timenow)_METACommunity_Round-$(round).csv", ExportDF)
    CSV.write("METACommunityPool.csv", METACommunityPool) ;
end


;

## 1. [Read and manipulate isolates files ]{style="color: #001C60;"} 

In [3]:
#| code-summary: "Read isolate trophic data CSV file"
filepath = "IsolateTrophicData_20250516.csv"
filedata = CSV.read(filepath, DataFrame) #Read CSV file at filepath
a = expandDataFrameProducts(filedata) #Separate products (e.g. pentose + hexose is made in two different entries/rows)
trophicdata = expandDataFrameSubstrates(a); #Separate substrates

In [4]:
#| code-summary: "Get information such as pathway and microbe names, IDs..."

speciesnames = string.(unique(trophicdata[!, :iso_index]))
ϕlevelsnames = unique(trophicdata[!, :Trophic_level])
pathnames = unique(trophicdata[!, :Activity])
substratenames = unique(trophicdata[!, :Substrate]) #20 substrates 
productnames = unique(trophicdata[!, :Product]) #13 products 
prodandsubsnames = unique(vcat(unique(trophicdata[!, :Substrate]), unique(trophicdata[!, :Product]))) #26 elements
produced_metacommunity = intersect(substratenames, productnames) #Six products that can be 'cross-fed'
l = length(trophicdata.Genome)

τ = 5 ;  #Trophic levels 

In [5]:
#| code-summary: "Create a binary dataframe on three levels: trophic levels, pathways, substrates and products"

ϕ_levels_collapsed_perm = getDATAFRAME(trophicdata, ϕlevelsnames, "Trophic_level")
paths_collapsed_perm = getDATAFRAME(trophicdata, pathnames, "Activity")
substrates_collapsed_perm = getDATAFRAME(trophicdata, prodandsubsnames, "Substrate");
products_collapsed_perm = getDATAFRAME(trophicdata, prodandsubsnames, "Product"); 
binarydataframes = [paths_collapsed_perm, substrates_collapsed_perm, products_collapsed_perm] ;

## 2. [Read and manipulate METACommunity files of the previous round]{style="color: #001C60;"} 

In [6]:
#| code-summary: "Import METACommunity and METACommunityPool files"

rm = 5 #Number of previous round
metacommunity_path = "2025.11.14_METACommunity_Round-5-AFTER.csv"
αMETACommunityDF = CSV.read(metacommunity_path, DataFrame) ; #METACommunity from the previous round.

impMETACommunity = [JSON3.read(Community, Vector{Int}) for Community in αMETACommunityDF.Members]
impMETACommunity = [string.(i) for i in impMETACommunity]
αMETACommunityDF.Members = impMETACommunity 
αMETACommunityDF.Members = [unique(αMETACommunityDF.Members[i]) for i in 1:length(αMETACommunityDF.Members)]

#Importing METACommunityPool, the file with the communities tested in all of the previous rounds
pool_path = "METACommunityPool.csv"
METACommunityPool = CSV.read(pool_path, DataFrame)

intermediatePool = [parse.(Int, JSON3.read(s)) for s in METACommunityPool.Members] #[JSON3.read(Community, Vector{Int}) for Community in METACommunityPool.Members]
memberstoString = [string.(i) for i in intermediatePool]
METACommunityPool.Members = memberstoString 
METACommunityPool.Members = [unique(METACommunityPool.Members[i]) for i in 1:length(METACommunityPool.Members)]

ηMETACommunityPool = vcat(METACommunityPool, αMETACommunityDF) ;

In [7]:
#| code-summary: "Selection of communities going to the roulette wheel"
#Calculate average community score (i.e fitness)
acs = mean(ηMETACommunityPool.CommunityScore)
wi = findall(x -> x > acs, ηMETACommunityPool.CommunityScore)
rouletteCommunities = ηMETACommunityPool[wi, :] ;

In [9]:
#| code-summary: "Re-assembly into the novel METACommunity"

#Roulette wheel only for the communities of the previous round:
#ηMETACommunity, motherID = reAssembly(αMETACommunityDF, ηMETACommunityPool) ;

#Roulette wheel of all communities in rouletteCommunities:
ηMETACommunity, motherID = reAssembly(rouletteCommunities, ηMETACommunityPool) ;

In [10]:
#| code-summary: "Get path coverage and functional integration (ψ) of the ηMETACommunity"
pathcoverage = getCoverage(ηMETACommunity, paths_collapsed_perm)
competition = getCompetition(ηMETACommunity, substrates_collapsed_perm)
complementarity = getComplementarity(ηMETACommunity, substrates_collapsed_perm, products_collapsed_perm)
ψ = mean.(complementarity) .* (1 .- mean.(competition)) ;

In [11]:
#| code-summary: "Export ηMETACommunity and updated METACommunityPool "

round = αMETACommunityDF.Round[1] + 1
exportMetacommunity(unique.(ηMETACommunity), ηMETACommunityPool, ψ, pathcoverage, motherID, round) ;